# Missing Data

## Overview

Missing data is not a nuisance — it carries information about *why* values are absent. Handling it correctly requires understanding the missingness mechanism before choosing a strategy.

**Missingness mechanisms:**

| Mechanism | Meaning | Safe to impute? |
|---|---|---|
| MCAR — Missing Completely at Random | Missingness unrelated to any variable | Yes |
| MAR — Missing at Random | Missingness depends on observed variables | Yes, with care |
| MNAR — Missing Not at Random | Missingness depends on the missing value itself | Problematic — model the missingness |

**Strategies:**
- Complete case analysis (listwise deletion): unbiased under MCAR only
- Single imputation: mean/median/mode, KNN, model-based
- Multiple imputation: generates m complete datasets, pools results
- Missingness indicator: add a binary flag column alongside imputed value

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import BayesianRidge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(42)
n = 400
# True complete data
elevation  = rng.uniform(50, 400, n)
nitrate    = rng.gamma(2, 2, n)
phosphorus = 0.6*nitrate + rng.normal(0, 0.5, n)
temp       = 15 - 0.02*elevation + rng.normal(0, 1, n)
richness   = 25 - 0.04*elevation - 0.8*nitrate + rng.normal(0, 3, n)
df_complete = pd.DataFrame(dict(elevation=elevation, nitrate=nitrate,
    phosphorus=phosphorus, temp=temp, richness=richness))
# Introduce MAR missingness: nitrate missing more at high elevation
p_miss_nitr = 0.05 + 0.2*(elevation - elevation.min())/(elevation.max()-elevation.min())
df = df_complete.copy()
df.loc[rng.random(n) < p_miss_nitr, 'nitrate'] = np.nan
df.loc[rng.random(n) < 0.08, 'phosphorus'] = np.nan
df.loc[rng.random(n) < 0.05, 'temp'] = np.nan
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nNitrate missing rate by elevation tertile:")
df['elev_tertile'] = pd.qcut(df['elevation'], 3, labels=['low','mid','high'])
print(df.groupby('elev_tertile')['nitrate'].apply(lambda x: x.isnull().mean()).round(3))

---
## Visualising Missingness Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# Missingness matrix
miss_matrix = df[['elevation','nitrate','phosphorus','temp']].isnull().astype(int)
im = axes[0].imshow(miss_matrix.T, aspect='auto', cmap='RdBu_r', vmin=0, vmax=1)
axes[0].set_yticks(range(4))
axes[0].set_yticklabels(['elevation','nitrate','phosphorus','temp'])
axes[0].set_xlabel('Observation index')
axes[0].set_title('Missingness Matrix (red = missing)')
plt.colorbar(im, ax=axes[0])
# Is missingness related to observed values? (MAR check)
nitrate_miss = df['nitrate'].isnull()
axes[1].hist(df.loc[~nitrate_miss, 'elevation'], bins=25, alpha=0.6,
             color='steelblue', density=True, label='Nitrate observed')
axes[1].hist(df.loc[nitrate_miss, 'elevation'], bins=25, alpha=0.6,
             color='#e74c3c', density=True, label='Nitrate missing')
axes[1].set_xlabel('Elevation')
axes[1].set_title('Nitrate Missingness by Elevation (MAR pattern)')
axes[1].legend(); plt.tight_layout(); plt.show()
from scipy import stats
t, p = stats.ttest_ind(df.loc[~nitrate_miss,'elevation'],
                        df.loc[nitrate_miss,'elevation'])
print(f"t-test: elevation differs by missingness? t={t:.3f}, p={p:.4f}")

---
## Imputation Strategies

In [ ]:
X_cols = ['elevation','nitrate','phosphorus','temp']
X = df[X_cols].copy()
y_col = df['richness']
strategies = {
    'Mean':        SimpleImputer(strategy='mean'),
    'Median':      SimpleImputer(strategy='median'),
    'KNN (k=5)':   KNNImputer(n_neighbors=5),
    'Iterative':   IterativeImputer(estimator=BayesianRidge(),
                                    max_iter=10, random_state=42),
}
print("Imputed nitrate statistics vs true complete data:")
print(f"{'Strategy':18s} {'Mean':>8} {'SD':>8} {'RMSE_vs_true':>14}")
true_nitr = df_complete['nitrate'].values
missing_mask = df['nitrate'].isnull()
for name, imp in strategies.items():
    X_imp = imp.fit_transform(X)
    nitr_imp = X_imp[:, 1]
    rmse = np.sqrt(np.mean((nitr_imp[missing_mask] - true_nitr[missing_mask])**2))
    print(f"{name:18s} {nitr_imp.mean():8.3f} {nitr_imp.std():8.3f} {rmse:14.3f}")

---
## Missingness Indicator Features

In [ ]:
# Add binary flag for whether nitrate was missing
# This captures the information in the missingness itself
df_feat = df.copy()
df_feat['nitrate_missing']    = df['nitrate'].isnull().astype(int)
df_feat['phosphorus_missing'] = df['phosphorus'].isnull().astype(int)
X_cols_with_ind = ['elevation','nitrate','phosphorus','temp',
                    'nitrate_missing','phosphorus_missing']
pipe_no_ind = Pipeline([
    ('imp', KNNImputer(n_neighbors=5)),
    ('scl', StandardScaler()),
    ('reg', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])
pipe_with_ind = Pipeline([
    ('imp', KNNImputer(n_neighbors=5)),
    ('scl', StandardScaler()),
    ('reg', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])
r2_no  = cross_val_score(pipe_no_ind,  df[X_cols], y_col, cv=5, scoring='r2').mean()
r2_yes = cross_val_score(pipe_with_ind, df_feat[X_cols_with_ind],
                          y_col, cv=5, scoring='r2').mean()
print(f"CV R2 without missingness indicators: {r2_no:.3f}")
print(f"CV R2 with    missingness indicators: {r2_yes:.3f}")
print("\nMissingness indicators help when missingness is MAR or MNAR")
print("and carries signal about the outcome")

In [ ]:
# Multiple imputation: pool results across m imputed datasets
m = 5  # number of imputed datasets
X_imp_datasets = []
imp_mi = IterativeImputer(max_iter=10, random_state=42, sample_posterior=True)
# Note: proper MI requires unique random states per imputation
for seed in range(m):
    imp_s = IterativeImputer(max_iter=10, random_state=seed, sample_posterior=True)
    X_imp_datasets.append(imp_s.fit_transform(df[X_cols]))
# Estimate coefficients from each imputed dataset, then pool (Rubin's rules)
import statsmodels.api as sm
coef_list, var_list = [], []
for X_imp in X_imp_datasets:
    X_sm = sm.add_constant(X_imp)
    m_fit = sm.OLS(y_col, X_sm).fit()
    coef_list.append(m_fit.params.values)
    var_list.append(m_fit.bse.values**2)
coef_pool = np.mean(coef_list, axis=0)
within_var = np.mean(var_list, axis=0)
between_var = np.var(coef_list, axis=0, ddof=1)
total_var = within_var + (1 + 1/m)*between_var
se_pool = np.sqrt(total_var)
print("Rubin's rules pooled estimates (Multiple Imputation):")
for i, name in enumerate(['const','elevation','nitrate','phosphorus','temp']):
    print(f"  {name:12s}: coef={coef_pool[i]:7.3f}, SE={se_pool[i]:.3f}")

---

## Common Pitfalls

**1. Defaulting to mean imputation without checking the missingness mechanism**  
Mean imputation is unbiased under MCAR but biased under MAR or MNAR. If nitrate is more likely to be missing at high-elevation sites, mean-imputing nitrate ignores its relationship with elevation and produces biased estimates. Always inspect missingness patterns before choosing a strategy.

**2. Imputing the target variable**  
Never impute the response variable. Rows where the outcome is missing should be excluded from modelling, not imputed. Imputing the target creates circular prediction and inflates model performance metrics.

**3. Fitting the imputer on training + test data**  
Imputation statistics (mean, kNN neighbours, iterative model weights) must be fitted on training data only and then applied to the test set. Fitting on the combined dataset leaks test-set information into the training-data imputation.

**4. Using single imputation and reporting confidence intervals as if the data were complete**  
Single imputation treats imputed values as known, underestimating uncertainty. Standard errors from a model fitted on singly-imputed data are too small. Use multiple imputation (with Rubin's rules for pooling) when valid confidence intervals matter.

**5. Not adding missingness indicator features when missingness is informative**  
When missingness is MAR or MNAR, the fact that a value is missing is itself informative. Discarding this signal by only imputing loses predictive power. Add a binary indicator column alongside the imputed value to let the model learn from the missingness pattern.
---
*python_methods_library - Samantha McGarrigle*